In [25]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "Stat-486-Final-Project"]

ROOT = next(
    (
        folder for folder in candidates
        if (folder / "results" / "model_predictions.csv").exists()
        and (folder / "data" / "processed" / "Broadway_Data_Clean.csv").exists()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        f"Could not find the project files. Notebook is running from: {cwd}"
    )

print("Project folder:", ROOT)

predictions = pd.read_csv(
    ROOT / "results" / "model_predictions.csv",
    parse_dates=["Week Ending"]
)
cleaned = pd.read_csv(
    ROOT / "data" / "processed" / "Broadway_Data_Clean.csv",
    parse_dates=["Week Ending"]
)

predictions["Show"] = predictions["Show"].astype(str).str.strip()
cleaned["Show"] = cleaned["Show"].astype(str).str.strip()
predictions["RF_Error"] = (
    predictions["RF_Pred"] - predictions["This Week Gross"]
)

Project folder: C:\Users\scott\OneDrive\Desktop\STAT 483\Stat-486-Final-Project


In [28]:
print("Output test")
print("ROOT:", ROOT)
print("Predictions:", predictions.shape)
print("Cleaned data:", cleaned.shape)

Output test
ROOT: C:\Users\scott\OneDrive\Desktop\STAT 483\Stat-486-Final-Project
Predictions: (1346, 11)
Cleaned data: (7347, 13)


In [26]:
#Load the prediction and cleaned data Reads the model’s saved predictions and the 
# cleaned Broadway dataset. It also standardizes show names and calculates the signed 
# Random Forest error. A positive error means the model predicted too high; 
# a negative error means it predicted too low.

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "Stat-486-Final-Project"]

ROOT = next(
    (
        folder for folder in candidates
        if (folder / "results" / "model_predictions.csv").exists()
        and (folder / "data" / "processed" / "Broadway_Data_Clean.csv").exists()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        f"Could not find the project files. Notebook is running from: {cwd}"
    )

print("Project folder:", ROOT)

Project folder: C:\Users\scott\OneDrive\Desktop\STAT 483\Stat-486-Final-Project


In [22]:
#Select the 20 largest errors and add theaters Finds the 20 predictions with
#  the largest absolute errors and matches each one to its theater in the 
# cleaned data. This gives us the shows, dates, actual grosses, predictions,
#  and error sizes for the biggest misses.
top20 = predictions.nlargest(20, "RF_AE").merge(
    cleaned[["Show", "Week Ending", "Theater"]].drop_duplicates(),
    on=["Show", "Week Ending"],
    how="left"
)

print(top20[[
    "Show", "Week Ending", "Theater",
    "This Week Gross", "RF_Pred", "RF_Error", "RF_AE"
]].to_string(index=False))

                             Show Week Ending                      Theater  This Week Gross      RF_Pred      RF_Error        RF_AE
               The Book of Mormon  2026-06-14       Eugene O'Neill Theatre       2238369.45 7.279152e+05 -1.510454e+06 1.510454e+06
                     Just in Time  2026-04-05 Circle in the Square Theatre        745152.25 2.255197e+06  1.510045e+06 1.510045e+06
Harry Potter and the Cursed Child  2026-05-17                Lyric Theatre        924640.53 2.259022e+06  1.334381e+06 1.334381e+06
                         Hamilton  2026-01-11      Richard Rodgers Theatre       2182545.00 3.491348e+06  1.308803e+06 1.308803e+06
            Every Brilliant Thing  2026-05-31               Hudson Theatre       1135322.20 2.394356e+06  1.259034e+06 1.259034e+06
                          Chicago  2026-05-10           Ambassador Theatre        556231.07 1.667902e+06  1.111671e+06 1.111671e+06
        Moulin Rouge! The Musical  2026-08-23        Al Hirschfeld Theatre  

In [20]:
#Look for repeated shows, theaters, and dates Counts which shows, theaters, 
# and reporting dates appear more than once among the 20 largest errors. 
# These repeats are clues to investigate; they don’t prove that a show, 
# theater, or date caused the errors.
print("Repeat appearances among the 20 largest errors:")

for column in ["Show", "Theater", "Week Ending"]:
    counts = top20[column].value_counts()
    repeated = counts[counts > 1]

    print(f"\n{column}:")
    print(repeated.to_string() if not repeated.empty else "No repeats")

Repeat appearances among the 20 largest errors:

Show:
Show
Harry Potter and the Cursed Child    4
The Lion King                        3
Just in Time                         2
The Book of Mormon                   2
Every Brilliant Thing                2

Theater:
Theater
Lyric Theatre                   4
Minskoff Theatre                3
Circle in the Square Theatre    2
Eugene O'Neill Theatre          2
Hudson Theatre                  2

Week Ending:
Week Ending
2026-01-11    4
2026-08-23    2
2026-04-05    2


In [9]:
# Check how much history each show has Finds how many weeks each show appears
#  in the full dataset and its first and last recorded dates. This helps show 
# whether a large error came from a show with extensive history or one with 
# fewer observations.
show_history = (
    cleaned[cleaned["Show"].isin(top20["Show"].unique())]
    .groupby("Show")
    .agg(
        weeks_in_dataset=("Week Ending", "nunique"),
        first_week=("Week Ending", "min"),
        last_week=("Week Ending", "max")
    )
    .sort_values("weeks_in_dataset", ascending=False)
)

print(show_history.to_string())

                                   weeks_in_dataset first_week  last_week
Show                                                                     
Chicago                                         253 2022-01-02 2026-11-01
MJ The Musical                                  253 2022-01-02 2026-11-01
Harry Potter and the Cursed Child               253 2022-01-02 2026-11-01
Hamilton                                        253 2022-01-02 2026-11-01
The Lion King                                   253 2022-01-02 2026-11-01
The Book of Mormon                              250 2022-01-02 2026-11-01
Moulin Rouge! The Musical                       244 2022-01-02 2026-08-30
Aladdin                                         219 2022-01-02 2026-11-01
Oh, Mary!                                       123 2024-06-30 2026-11-01
Just in Time                                     83 2025-04-06 2026-11-01
Chess                                            36 2025-10-19 2026-06-21
Every Brilliant Thing                 

In [10]:
# Flag unusually high grosses and unusually large errors within each show 
# Uses the IQR rule to compare each show’s grosses with its own gross history,
#  and each show’s prediction errors with its own test-period errors. 
# The rule flags values beyond the usual range.
#  A flag means “unusual,” not necessarily incorrect.
gross_stats = cleaned.groupby("Show").agg(
    gross_weeks=("This Week Gross", "count"),
    gross_q1=("This Week Gross", lambda x: x.quantile(0.25)),
    gross_q3=("This Week Gross", lambda x: x.quantile(0.75))
)
gross_stats["gross_iqr"] = gross_stats["gross_q3"] - gross_stats["gross_q1"]
gross_stats["gross_lower"] = (
    gross_stats["gross_q1"] - 1.5 * gross_stats["gross_iqr"]
)
gross_stats["gross_upper"] = (
    gross_stats["gross_q3"] + 1.5 * gross_stats["gross_iqr"]
)

error_stats = predictions.groupby("Show").agg(
    test_weeks=("RF_AE", "count"),
    error_q1=("RF_AE", lambda x: x.quantile(0.25)),
    error_q3=("RF_AE", lambda x: x.quantile(0.75))
)
error_stats["error_upper"] = (
    error_stats["error_q3"]
    + 1.5 * (error_stats["error_q3"] - error_stats["error_q1"])
)

top20 = top20.join(gross_stats, on="Show")
top20 = top20.join(error_stats, on="Show")

top20["Gross unusual for this show"] = (
    (top20["This Week Gross"] < top20["gross_lower"])
    | (top20["This Week Gross"] > top20["gross_upper"])
)
top20["Error unusual for this show"] = (
    top20["RF_AE"] > top20["error_upper"]
)

In [11]:
# Compare actuals and predictions with each show’s pre-2026 gross range
# Finds each show’s minimum and maximum gross before the 2026 test period.
# It then checks whether the actual gross or Random Forest prediction falls
#  outside that historical range. This helps identify cases where the show’s 
# actual result or the model’s prediction is beyond what that show had 
# previously recorded.
train_ranges = (
    cleaned.loc[cleaned["Week Ending"] < pd.Timestamp("2026-01-04")]
    .groupby("Show")
    .agg(
        train_min=("This Week Gross", "min"),
        train_max=("This Week Gross", "max")
    )
)

top20 = top20.join(train_ranges, on="Show")

top20["Actual outside prior range"] = (
    (top20["This Week Gross"] < top20["train_min"])
    | (top20["This Week Gross"] > top20["train_max"])
)
top20["Prediction outside prior range"] = (
    (top20["RF_Pred"] < top20["train_min"])
    | (top20["RF_Pred"] > top20["train_max"])
)

In [12]:
# Print the top 20 cases in a readable table Displays the largest misses 
# alongside the show, theater, date, actual gross, prediction, training
#  range, and outlier flags. Dollar values are formatted for easier reading.

columns = [
    "Show", "Week Ending", "Theater",
    "This Week Gross", "RF_Pred", "RF_Error", "RF_AE",
    "train_min", "train_max",
    "Actual outside prior range", "Prediction outside prior range",
    "Gross unusual for this show", "Error unusual for this show"
]

money_columns = [
    "This Week Gross", "RF_Pred", "RF_Error", "RF_AE",
    "train_min", "train_max"
]
format_money = {
    column: lambda value: f"${value:,.0f}"
    for column in money_columns
}

print(
    top20[columns].to_string(
        index=False,
        formatters=format_money
    )
)

                             Show Week Ending                      Theater This Week Gross    RF_Pred    RF_Error      RF_AE  train_min  train_max  Actual outside prior range  Prediction outside prior range  Gross unusual for this show  Error unusual for this show
               The Book of Mormon  2026-06-14       Eugene O'Neill Theatre      $2,238,369   $727,915 $-1,510,454 $1,510,454   $651,270 $1,803,112                        True                           False                         True                         True
                     Just in Time  2026-04-05 Circle in the Square Theatre        $745,152 $2,255,197  $1,510,045 $1,510,045   $808,299 $1,640,274                        True                            True                        False                         True
Harry Potter and the Cursed Child  2026-05-17                Lyric Theatre        $924,641 $2,259,022  $1,334,381 $1,334,381   $710,462 $3,709,875                       False                           Fals

In [13]:
#Count unusually large errors across all test predictions Applies the 
# show-specific IQR error threshold to the full test set, not only the 
# top 20. It reports the number and percentage of unusually large show-week 
# errors, how many shows they affect, and how many flagged errors each 
# show has.
all_errors = predictions.join(error_stats, on="Show")

all_errors["Error unusual for this show"] = (
    all_errors["RF_AE"] > all_errors["error_upper"]
)

unusual_errors = all_errors[all_errors["Error unusual for this show"]]

print(
    f"Unusually high show-week errors: "
    f"{len(unusual_errors)} of {len(predictions)} "
    f"({len(unusual_errors) / len(predictions):.1%})"
)
print(
    f"Shows with at least one unusually high error: "
    f"{unusual_errors['Show'].nunique()} of {predictions['Show'].nunique()}"
)

print("\nShows with unusual errors:")
print(
    unusual_errors.groupby("Show").agg(
        unusual_weeks=("RF_AE", "size"),
        largest_error=("RF_AE", "max")
    ).sort_values("unusual_weeks", ascending=False).to_string()
)

Unusually high show-week errors: 74 of 1346 (5.5%)
Shows with at least one unusually high error: 34 of 58

Shows with unusual errors:
                                   unusual_weeks  largest_error
Show                                                           
Moulin Rouge! The Musical                      5   1.110441e+06
Just in Time                                   5   1.510045e+06
The Book of Mormon                             5   1.510454e+06
The Great Gatsby                               4   4.636410e+05
Two Strangers (                                4   2.292654e+05
Cats: The Jellicle Ball                        4   4.873591e+05
Hadestown                                      4   3.891319e+05
Chicago                                        4   1.111671e+06
Every Brilliant Thing                          3   1.259034e+06
Aladdin                                        3   1.035976e+06
Operation Mincemeat                            3   4.591084e+05
The Lion King                     